Imports & Dataset Paths, Dataset Split


In [ ]:


import os
import shutil
import random
from sklearn.model_selection import train_test_split

DATASET_PATH = '/../Images of Nutrient Deficient Banana Plant Leaves/Images of Nutrient Deficient Banana Plant Leaves/Augmented Images of Nutrient Deficient Banana leaves'  # Change this to your dataset path
SAVE_PATH = '/../split_dataset'

# Count images in each split
def count_images():
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(SAVE_PATH, split)
        total_images = 0
        print(f"\n{split.upper()} SET:")
        for class_name in os.listdir(split_path):
            class_path = os.path.join(split_path, class_name)
            if os.path.isdir(class_path):
                num_images = len(os.listdir(class_path))
                total_images += num_images
                print(f"{class_name}: {num_images} images")
        print(f"Total {split} images: {total_images}")

count_images()

Corrupt Image Detection & Removal

In [ ]:
import os
from PIL import Image
DATASET_PATH = '/../split_dataset'
def check_corrupt_images(directory):
    for folder in ['train', 'val','test']:  # Modify if you have more subfolders
        folder_path = os.path.join(directory, folder)
        for class_folder in os.listdir(folder_path):
            class_path = os.path.join(folder_path, class_folder)
            if not os.path.isdir(class_path):
                continue
            for filename in os.listdir(class_path):
                file_path = os.path.join(class_path, filename)
                try:
                    with Image.open(file_path) as img:
                        img.verify()  # Verify the image
                except (IOError, SyntaxError):
                    print(f"Corrupt image found and removed: {file_path}")
                    os.remove(file_path)  # Delete corrupt file

check_corrupt_images(DATASET_PATH)


Image Format Standardization(JPG Format)

In [ ]:
from PIL import Image

def convert_images_to_jpg(directory):
    for folder in ['train', 'val','test']:  # Modify if needed
        folder_path = os.path.join(directory, folder)
        for class_folder in os.listdir(folder_path):
            class_path = os.path.join(folder_path, class_folder)
            if not os.path.isdir(class_path):
                continue
            for filename in os.listdir(class_path):
                file_path = os.path.join(class_path, filename)
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):  # Convert only non-JPG/PNG files
                    try:
                        img = Image.open(file_path)
                        rgb_img = img.convert('RGB')  # Convert to RGB if needed
                        new_file_path = os.path.splitext(file_path)[0] + ".jpg"
                        rgb_img.save(new_file_path, 'JPEG')
                        os.remove(file_path)  # Remove the original file
                        print(f"Converted and saved: {new_file_path}")
                    except Exception as e:
                        print(f"Error converting {file_path}: {e}")

convert_images_to_jpg(DATASET_PATH)


ResNet50V2

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt
import os



# Define dataset path
DATASET_PATH = '/../split_dataset'
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCHS = 30

# Data generators
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

def load_data():
    train_data = train_datagen.flow_from_directory(
        os.path.join(DATASET_PATH, 'train'),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
    val_data = val_datagen.flow_from_directory(
        os.path.join(DATASET_PATH, 'val'),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
    return train_data, val_data

# Load data
train_data, val_data = load_data()

# Create ResNet50V2 model
def create_resnet_model():
    base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.2)(x)
    outputs = Dense(train_data.num_classes, activation='softmax')(x)
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Train model
resnet_model = create_resnet_model()
history = resnet_model.fit(train_data, validation_data=val_data, epochs=EPOCHS)

# Save model
resnet_model.save('/../ResNet50V2_model.h5')

# Plot validation accuracy
plt.plot(history.history['val_accuracy'], label='ResNet50V2')
plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.title('Validation Accuracy - ResNet50V2')
plt.show()




MobileNetV2

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import os



# Common configuration
DATASET_PATH = '/../split_dataset'
BATCH_SIZE = 32
EPOCHS = 30
CLASS_MODE = 'categorical'
ORIGINAL_SIZE = (225, 225)  # Your original image size

# Callbacks for all models
def get_callbacks():
    return [
        EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
        ReduceLROnPlateau(factor=0.2, patience=3, monitor='val_loss')
    ]

# Enhanced Data Generator with Smart Resizing
def create_data_generator(model_name, augment=False):
    model_config = {
        'MobileNetV2': {
            'preprocess': tf.keras.applications.mobilenet_v2.preprocess_input,
            'target_size': (224, 224),
            'interpolation': 'bilinear'  # For slight downscaling 225→224
        }
    }

    config = model_config[model_name]

    if augment:
        return ImageDataGenerator(
            preprocessing_function=config['preprocess'],
            rotation_range=15,  # Slightly reduced for resized images
            width_shift_range=0.15,
            height_shift_range=0.15,
            shear_range=0.15,
            zoom_range=0.15,
            horizontal_flip=True,
            fill_mode='reflect'
        )
    else:
        return ImageDataGenerator(
            preprocessing_function=config['preprocess']
        )

# Plotting function
def plot_history(history, model_name):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{model_name} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{model_name} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Model Creation Functions with 225×225 Handling
def create_mobilenetv2(num_classes):
    base_model = tf.keras.applications.MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)  # Model expects 224×224
    )

    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(*ORIGINAL_SIZE, 3))  # Your 225×225 input
    # Resize layer will handle 225→224 conversion
    x = tf.keras.layers.Resizing(224, 224, interpolation='bilinear')(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def create_inceptionv3(num_classes):
    base_model = tf.keras.applications.InceptionV3(
        weights='imagenet',
        include_top=False,
        input_shape=(299, 299, 3)  # Model expects 299×299
    )

    base_model.trainable = True
    for layer in base_model.layers[:-15]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(*ORIGINAL_SIZE, 3))  # Your 225×225 input
    # Resize layer will handle 225→299 conversion
    x = tf.keras.layers.Resizing(299, 299, interpolation='bicubic')(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=2e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def create_efficientnetb3(num_classes):
    base_model = tf.keras.applications.EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=(300, 300, 3),  # Model expects 300×300
        drop_connect_rate=0.4
    )

    base_model.trainable = True
    for layer in base_model.layers[:-10]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(*ORIGINAL_SIZE, 3))  # Your 225×225 input
    # Resize layer will handle 225→300 conversion
    x = tf.keras.layers.Resizing(300, 300, interpolation='bicubic')(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.6)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def create_resnet50v2(num_classes):
    base_model = tf.keras.applications.ResNet50V2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)  # Model expects 224×224
    )

    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(*ORIGINAL_SIZE, 3))  # Your 225×225 input
    # Resize layer will handle 225→224 conversion
    x = tf.keras.layers.Resizing(224, 224, interpolation='bilinear')(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Training Execution
def train_model(model_creator, model_name):
    print(f"\nTraining {model_name}...")

    train_datagen = create_data_generator(model_name, augment=True)
    val_datagen = create_data_generator(model_name, augment=False)

    train_data = train_datagen.flow_from_directory(
        os.path.join(DATASET_PATH, 'train'),
        target_size=ORIGINAL_SIZE,  # Use original 225×225
        batch_size=BATCH_SIZE,
        class_mode=CLASS_MODE
    )

    val_data = val_datagen.flow_from_directory(
        os.path.join(DATASET_PATH, 'val'),
        target_size=ORIGINAL_SIZE,  # Use original 225×225
        batch_size=BATCH_SIZE,
        class_mode=CLASS_MODE
    )

    # Get the number of classes from the training data
    num_classes = train_data.num_classes

    # Pass num_classes to the model creator
    model = model_creator(num_classes)
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS,
        callbacks=get_callbacks()
    )

    model.save(f'/../Optimized_{model_name}_225input_model.h5')
    plot_history(history, model_name)
    return history

# Train all models
models = {
    'MobileNetV2': create_mobilenetv2,

}

for name, creator in models.items():
    train_model(creator, name)

print("All models trained and saved!")

InceptionV3

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import os

EPOCHS = 30
CLASS_MODE = 'categorical'
ORIGINAL_SIZE = (225, 225)  # Your original image size
BATCH_SIZE = 32  # Define your batch size
DATASET_PATH = '/../split_dataset'  # Update this to your dataset path

# Callbacks for all models
def get_callbacks():
    return [
        EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
        ReduceLROnPlateau(factor=0.2, patience=3, monitor='val_loss')
    ]

# Enhanced Data Generator with Smart Resizing
def create_data_generator(model_name, augment=False):
    model_config = {

        'InceptionV3': {
            'preprocess': tf.keras.applications.inception_v3.preprocess_input,
            'target_size': (299, 299),
            'interpolation': 'bilinear'  # For upscaling 225→299
        }
    }

    config = model_config[model_name]

    if augment:
        return ImageDataGenerator(
            preprocessing_function=config['preprocess'],
            rotation_range=30,  # Increased rotation for more variation
            width_shift_range=0.2,
            height_shift_range=0.2,
            shear_range=0.2,
            zoom_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest'  # 'nearest' to avoid black padding during transformations
        )
    else:
        return ImageDataGenerator(
            preprocessing_function=config['preprocess']
        )

# Plotting function
def plot_history(history, model_name):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{model_name} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{model_name} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Model Creation Functions with 225×225 Handling
def create_mobilenetv2(num_classes):
    base_model = tf.keras.applications.MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)  # Model expects 224×224
    )

    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(*ORIGINAL_SIZE, 3))  # Your 225×225 input
    # Resize layer will handle 225→224 conversion
    x = tf.keras.layers.Resizing(224, 224, interpolation='bilinear')(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def create_inceptionv3(num_classes):
    base_model = tf.keras.applications.InceptionV3(
        weights='imagenet',
        include_top=False,
        input_shape=(299, 299, 3)  # Model expects 299×299
    )

    base_model.trainable = True
    for layer in base_model.layers[:-15]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(*ORIGINAL_SIZE, 3))  # Your 225×225 input
    # Resize layer will handle 225→299 conversion
    x = tf.keras.layers.Resizing(299, 299, interpolation='bilinear')(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=2e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def create_resnet50v2(num_classes):
    base_model = tf.keras.applications.ResNet50V2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)  # Model expects 224×224
    )

    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(*ORIGINAL_SIZE, 3))  # Your 225×225 input
    # Resize layer will handle 225→224 conversion
    x = tf.keras.layers.Resizing(224, 224, interpolation='bilinear')(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def create_resnet152v2(num_classes):
    base_model = tf.keras.applications.ResNet152V2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)  # Model expects 224×224
    )

    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(*ORIGINAL_SIZE, 3))  # Your 225×225 input
    # Resize layer will handle 225→224 conversion
    x = tf.keras.layers.Resizing(224, 224, interpolation='bilinear')(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Training Execution
def train_model(model_creator, model_name):
    print(f"\nTraining {model_name}...")

    train_datagen = create_data_generator(model_name, augment=True)
    val_datagen = create_data_generator(model_name, augment=False)

    train_data = train_datagen.flow_from_directory(
        os.path.join(DATASET_PATH, 'train'),
        target_size=ORIGINAL_SIZE,  # Use original 225×225
        batch_size=BATCH_SIZE,
        class_mode=CLASS_MODE
    )

    val_data = val_datagen.flow_from_directory(
        os.path.join(DATASET_PATH, 'val'),
        target_size=ORIGINAL_SIZE,  # Use original 225×225
        batch_size=BATCH_SIZE,
        class_mode=CLASS_MODE
    )

    # Get the number of classes from the training data
    num_classes = train_data.num_classes

    # Pass num_classes to the model creator
    model = model_creator(num_classes)
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS,
        callbacks=get_callbacks()
    )

    model.save(f'/../Optimized_{model_name}_225input_model.h5')
    plot_history(history, model_name)
    return history

# Train the selected models
models = {

    'InceptionV3': create_inceptionv3,
    }

for name, creator in models.items():
    train_model(creator, name)

print("All selected models trained and saved!")

